# Name That LLM — Nested Stack V4

Four text/style classifiers feed a regularized multinomial logistic combiner. The combiner is trained on predictions made without training on each passage's prompt group.

Nested development balanced accuracy: **37.8213%**. Frozen related-prompt comparison: **39.5247%**, versus **38.1186%** for v3. Available labeled data is reused; these are not independent test estimates.

CPU, internet disabled, no external data or pretrained weights. Test IDs retain their original order; predictions are independent of other test passages.

In [ ]:
import os
for key in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "LOKY_MAX_CPU_COUNT"):
    os.environ[key] = "2"


In [ ]:
import json
from pathlib import Path
import sys
sources = json.loads('{"__init__.py": "\\"\\"\\"Reproducible baselines for Name That LLM.\\"\\"\\"\\n", "baseline.py": "\\"\\"\\"Shared training and submission code, also embedded in the Kaggle notebook.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport logging\\nimport platform\\nimport time\\nimport warnings\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport scipy\\nimport sklearn\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics import balanced_accuracy_score, confusion_matrix, recall_score\\nfrom sklearn.model_selection import GroupKFold\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer\\nfrom sklearn.svm import LinearSVC\\n\\nLOGGER = logging.getLogger(\\"name_that_llm\\")\\nLABELS = tuple(sorted((\\n    \\"gpt-5.5\\", \\"gpt-5.6-luna\\", \\"gpt-5.6-sol\\", \\"gpt-5.6-terra\\",\\n    \\"gpt-6-astra\\", \\"gemini-3.1-pro-preview\\", \\"gemini-3.6-flash\\",\\n    \\"gemini-3.7-flash\\", \\"gemini-3.8-flash\\", \\"claude-fable-5\\",\\n    \\"claude-opus-4-6\\", \\"claude-opus-4-8\\", \\"claude-opus-5\\",\\n    \\"claude-sonnet-5\\", \\"qwen3.8-max\\", \\"glm-5.3\\",\\n    \\"glm-5.3-flash\\", \\"kimi-k3\\",\\n)))\\n\\n\\n@dataclass(frozen=True)\\nclass BaselineConfig:\\n    \\"\\"\\"Fixed before inspecting any validation results.\\"\\"\\"\\n\\n    variant: str = \\"char\\"\\n    seed: int = 42\\n    n_splits: int = 5\\n    c: float = 1.0\\n    min_df: int = 2\\n    char_max_features: int = 250_000\\n    word_max_features: int = 100_000\\n    max_iter: int = 10_000\\n    tolerance: float = 1e-4\\n\\n\\ndef validate_frame(frame: pd.DataFrame, *, training: bool) -> None:\\n    required = [\\"id\\", \\"text\\"] + ([\\"prompt_id\\", \\"model\\"] if training else [])\\n    missing = set(required) - set(frame.columns)\\n    if missing:\\n        raise ValueError(f\\"Missing columns: {sorted(missing)}\\")\\n    if frame.empty:\\n        raise ValueError(\\"Input must contain at least one passage.\\")\\n    for column in required:\\n        if frame[column].isna().any() or not frame[column].map(\\n            lambda value: isinstance(value, str) and bool(value.strip())\\n        ).all():\\n            raise ValueError(f\\"Column {column!r} contains missing, empty or non-string values.\\")\\n    if frame[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Passage IDs must be unique.\\")\\n    if training and set(frame[\\"model\\"]) != set(LABELS):\\n        raise ValueError(\\"Training labels must match all 18 competition labels.\\")\\n\\n\\ndef load_data(data_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:\\n    # Explicit string types retain leading zeros and the exact CSV-decoded text.\\n    train = pd.read_csv(data_dir / \\"train.csv\\", dtype=str, keep_default_na=False)\\n    test = pd.read_csv(data_dir / \\"test.csv\\", dtype=str, keep_default_na=False)\\n    validate_frame(train, training=True)\\n    validate_frame(test, training=False)\\n    return train, test\\n\\n\\ndef build_pipeline(config: BaselineConfig) -> Pipeline:\\n    if config.variant not in {\\"char\\", \\"char_word\\"}:\\n        raise ValueError(f\\"Unknown baseline variant: {config.variant}\\")\\n    common = dict(min_df=config.min_df, sublinear_tf=True, dtype=np.float32)\\n    features = [(\\"char\\", TfidfVectorizer(\\n        analyzer=\\"char\\", ngram_range=(3, 5), lowercase=False,\\n        strip_accents=None, max_features=config.char_max_features, **common,\\n    ))]\\n    if config.variant == \\"char_word\\":\\n        features.append((\\"word\\", TfidfVectorizer(\\n            analyzer=\\"word\\", ngram_range=(1, 2), lowercase=True,\\n            strip_accents=None, stop_words=None,\\n            max_features=config.word_max_features, **common,\\n        )))\\n    return Pipeline([\\n        (\\"features\\", FeatureUnion(features)),\\n        (\\"normalize\\", Normalizer()),\\n        (\\"classifier\\", LinearSVC(\\n            C=config.c, dual=True, random_state=config.seed,\\n            max_iter=config.max_iter, tol=config.tolerance,\\n        )),\\n    ])\\n\\n\\ndef make_folds(train: pd.DataFrame, config: BaselineConfig) -> list[tuple[np.ndarray, np.ndarray]]:\\n    validate_frame(train, training=True)\\n    splitter = GroupKFold(\\n        n_splits=config.n_splits, shuffle=True, random_state=config.seed,\\n    )\\n    folds = list(splitter.split(train[\\"text\\"], train[\\"model\\"], groups=train[\\"prompt_id\\"]))\\n    counts = np.zeros(len(train), dtype=int)\\n    for fitting, held_out in folds:\\n        if set(train.iloc[fitting][\\"prompt_id\\"]) & set(train.iloc[held_out][\\"prompt_id\\"]):\\n            raise ValueError(\\"A prompt appears in both sides of a fold.\\")\\n        for indexes in (fitting, held_out):\\n            if set(train.iloc[indexes][\\"model\\"]) != set(LABELS):\\n                raise ValueError(\\"Every fold must contain all 18 labels on both sides.\\")\\n        counts[held_out] += 1\\n    if not np.all(counts == 1):\\n        raise ValueError(\\"Each passage must be held out exactly once.\\")\\n    return folds\\n\\n\\ndef fit_checked(pipeline: Pipeline, text: pd.Series, labels: pd.Series) -> Pipeline:\\n    with warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        pipeline.fit(text, labels)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(\\"LinearSVC did not converge; inspect the run log before proceeding.\\")\\n    return pipeline\\n\\n\\ndef validate_submission(\\n    test: pd.DataFrame, submission: pd.DataFrame,\\n) -> None:\\n    validate_frame(test, training=False)\\n    if list(submission.columns) != [\\"id\\", \\"model\\"]:\\n        raise ValueError(\\"Submission columns must be exactly id, model.\\")\\n    if len(submission) != len(test):\\n        raise ValueError(\\"Submission row count differs from the runtime test set.\\")\\n    if submission.isna().any().any() or submission[\\"id\\"].duplicated().any():\\n        raise ValueError(\\"Submission contains missing values or duplicate IDs.\\")\\n    if submission[\\"id\\"].tolist() != test[\\"id\\"].tolist():\\n        raise ValueError(\\"Submission IDs must match runtime test IDs in order.\\")\\n    if not set(submission[\\"model\\"]).issubset(LABELS):\\n        raise ValueError(\\"Submission contains an unknown model label.\\")\\n\\n\\ndef predict_submission(pipeline: Pipeline, test: pd.DataFrame) -> pd.DataFrame:\\n    validate_frame(test, training=False)\\n    predictions = pipeline.predict(test[\\"text\\"])\\n    submission = pd.DataFrame({\\"id\\": test[\\"id\\"].to_numpy(), \\"model\\": predictions})\\n    validate_submission(test, submission)\\n    return submission\\n\\n\\ndef file_sha256(path: Path) -> str:\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef runtime_versions() -> dict[str, str]:\\n    return {\\n        \\"python\\": platform.python_version(), \\"numpy\\": np.__version__,\\n        \\"pandas\\": pd.__version__, \\"scipy\\": scipy.__version__,\\n        \\"scikit_learn\\": sklearn.__version__,\\n    }\\n\\n\\nclass BaselineExperiment:\\n    \\"\\"\\"Compare two predeclared candidates on a single fixed prompt split.\\"\\"\\"\\n\\n    def __init__(self, data_dir: Path, output_dir: Path):\\n        self.data_dir = data_dir\\n        self.output_dir = output_dir\\n\\n    def run(self) -> dict:\\n        self.output_dir.mkdir(parents=True, exist_ok=True)\\n        train, test = load_data(self.data_dir)\\n        configs = [BaselineConfig(variant=v) for v in (\\"char\\", \\"char_word\\")]\\n        folds = make_folds(train, configs[0])\\n        assignment = train[[\\"id\\", \\"prompt_id\\"]].copy()\\n        assignment[\\"fold\\"] = -1\\n        for fold, (_, held_out) in enumerate(folds):\\n            assignment.loc[assignment.index[held_out], \\"fold\\"] = fold\\n        assignment.to_csv(self.output_dir / \\"folds.csv\\", index=False)\\n        summary = {\\n            \\"selection_metric\\": \\"pooled_oof_balanced_accuracy\\",\\n            \\"tie_break\\": \\"char (simpler candidate)\\",\\n            \\"versions\\": runtime_versions(),\\n            \\"train_sha256\\": file_sha256(self.data_dir / \\"train.csv\\"),\\n            \\"train_rows\\": len(train), \\"prompt_count\\": train[\\"prompt_id\\"].nunique(),\\n            \\"labels\\": list(LABELS), \\"candidates\\": {},\\n            \\"limitations\\": [\\n                \\"Prompt groups can still share related topics.\\",\\n                \\"Features were motivated by a label-visible review of all training data.\\",\\n                \\"Selecting the better of two candidates can make its CV estimate optimistic.\\",\\n                \\"Visible test predictions are a format check, not a scored evaluation.\\",\\n            ],\\n        }\\n        for config in configs:\\n            oof = np.empty(len(train), dtype=object)\\n            fold_metrics = []\\n            candidate_dir = self.output_dir / config.variant\\n            candidate_dir.mkdir(exist_ok=True)\\n            for fold, (fitting, held_out) in enumerate(folds):\\n                LOGGER.info(\\"Start %s fold=%d train=%d valid=%d\\",\\n                            config.variant, fold, len(fitting), len(held_out))\\n                started = time.perf_counter()\\n                pipeline = fit_checked(\\n                    build_pipeline(config), train.iloc[fitting][\\"text\\"],\\n                    train.iloc[fitting][\\"model\\"],\\n                )\\n                oof[held_out] = pipeline.predict(train.iloc[held_out][\\"text\\"])\\n                score = balanced_accuracy_score(train.iloc[held_out][\\"model\\"], oof[held_out])\\n                feature_sizes = {\\n                    name: len(vectorizer.vocabulary_)\\n                    for name, vectorizer in pipeline.named_steps[\\"features\\"].transformer_list\\n                }\\n                result = {\\n                    \\"fold\\": fold, \\"balanced_accuracy\\": float(score),\\n                    \\"train_rows\\": len(fitting), \\"validation_rows\\": len(held_out),\\n                    \\"seconds\\": time.perf_counter() - started,\\n                    \\"feature_counts\\": feature_sizes,\\n                    \\"solver_iterations\\": int(pipeline.named_steps[\\"classifier\\"].n_iter_),\\n                }\\n                fold_metrics.append(result)\\n                LOGGER.info(\\"Done %s %s\\", config.variant, json.dumps(result))\\n            predictions = train[[\\"id\\", \\"prompt_id\\", \\"model\\"]].copy()\\n            predictions[\\"fold\\"] = assignment[\\"fold\\"]\\n            predictions[\\"prediction\\"] = oof\\n            predictions.to_csv(candidate_dir / \\"oof_predictions.csv\\", index=False)\\n            matrix = confusion_matrix(train[\\"model\\"], oof, labels=LABELS)\\n            pd.DataFrame(matrix, index=LABELS, columns=LABELS).rename_axis(\\"true_model\\").to_csv(\\n                candidate_dir / \\"confusion_matrix.csv\\"\\n            )\\n            recalls = recall_score(train[\\"model\\"], oof, labels=LABELS, average=None)\\n            per_class = pd.DataFrame({\\n                \\"model\\": LABELS, \\"recall\\": recalls,\\n                \\"support\\": matrix.sum(axis=1),\\n                \\"correct\\": matrix.diagonal(),\\n            })\\n            per_class.to_csv(candidate_dir / \\"per_class_recall.csv\\", index=False)\\n            scores = [result[\\"balanced_accuracy\\"] for result in fold_metrics]\\n            summary[\\"candidates\\"][config.variant] = {\\n                \\"config\\": asdict(config),\\n                \\"pooled_oof_balanced_accuracy\\": float(balanced_accuracy_score(train[\\"model\\"], oof)),\\n                \\"fold_mean\\": float(np.mean(scores)),\\n                \\"fold_standard_deviation\\": float(np.std(scores, ddof=1)),\\n                \\"folds\\": fold_metrics,\\n                \\"per_class\\": per_class.to_dict(orient=\\"records\\"),\\n            }\\n        selected = max(\\n            summary[\\"candidates\\"],\\n            key=lambda name: summary[\\"candidates\\"][name][\\"pooled_oof_balanced_accuracy\\"],\\n        )\\n        summary[\\"selected_variant\\"] = selected\\n        # Write the comparison before the separate full-data training step.\\n        self._write_summary(summary)\\n        LOGGER.info(\\"Selected %s; refitting on all %d training rows\\", selected, len(train))\\n        config = BaselineConfig(**summary[\\"candidates\\"][selected][\\"config\\"])\\n        started = time.perf_counter()\\n        pipeline = fit_checked(build_pipeline(config), train[\\"text\\"], train[\\"model\\"])\\n        submission = predict_submission(pipeline, test)\\n        submission_path = self.output_dir / \\"submission.parquet\\"\\n        submission.to_parquet(submission_path, index=False)\\n        validate_submission(test, pd.read_parquet(submission_path))\\n        summary[\\"full_fit_and_preview_seconds\\"] = time.perf_counter() - started\\n        summary[\\"visible_test_rows\\"] = len(test)\\n        summary[\\"preview_submission_sha256\\"] = file_sha256(submission_path)\\n        self._write_summary(summary)\\n        LOGGER.info(\\"Preview submission verified: rows=%d; this is not a leaderboard score.\\", len(test))\\n        return summary\\n\\n    def _write_summary(self, summary: dict) -> None:\\n        (self.output_dir / \\"metrics.json\\").write_text(json.dumps(summary, indent=2) + \\"\\\\n\\")\\n", "boosting_v3.py": "\\"\\"\\"Three fixed CPU LightGBM recipes using the existing style measurements.\\n\\nDART motivation: https://arxiv.org/html/1505.01866v1.\\nOfficial implementation: https://github.com/lightgbm-org/LightGBM.\\nThis bounded experiment is not a reproduction or a state-of-the-art claim.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom lightgbm import LGBMClassifier\\nfrom sklearn.pipeline import Pipeline\\n\\nfrom name_that_llm.stylometry import StyleFeatures\\n\\n\\nBOOSTING_CANDIDATES = {\\n    \\"lightgbm7\\": {\\n        \\"boosting_type\\": \\"gbdt\\",\\n        \\"n_estimators\\": 400,\\n        \\"learning_rate\\": .035,\\n        \\"num_leaves\\": 7,\\n        \\"max_depth\\": 4,\\n        \\"min_child_samples\\": 25,\\n        \\"reg_lambda\\": 3.,\\n        \\"colsample_bytree\\": .8,\\n    },\\n    \\"lightgbm15\\": {\\n        \\"boosting_type\\": \\"gbdt\\",\\n        \\"n_estimators\\": 300,\\n        \\"learning_rate\\": .04,\\n        \\"num_leaves\\": 15,\\n        \\"max_depth\\": 5,\\n        \\"min_child_samples\\": 30,\\n        \\"reg_lambda\\": 10.,\\n        \\"colsample_bytree\\": .7,\\n    },\\n    \\"dart7\\": {\\n        \\"boosting_type\\": \\"dart\\",\\n        \\"n_estimators\\": 300,\\n        \\"learning_rate\\": .08,\\n        \\"num_leaves\\": 7,\\n        \\"max_depth\\": 4,\\n        \\"min_child_samples\\": 25,\\n        \\"reg_lambda\\": 3.,\\n        \\"colsample_bytree\\": .8,\\n        \\"drop_rate\\": .03,\\n        \\"skip_drop\\": .5,\\n        \\"max_drop\\": 30,\\n    },\\n}\\n\\nBOOSTING_COMMON = {\\n    \\"objective\\": \\"multiclass\\",\\n    \\"n_jobs\\": 2,\\n    \\"random_state\\": 42,\\n    \\"deterministic\\": True,\\n    \\"force_col_wise\\": True,\\n    \\"verbosity\\": -1,\\n    \\"subsample\\": 1.,\\n    \\"subsample_freq\\": 0,\\n    \\"max_bin\\": 127,\\n    \\"reg_alpha\\": 0.,\\n}\\n\\n\\ndef build_boosting_pipeline(name):\\n    \\"\\"\\"Return a fresh pipeline whose only learned component sees fit rows.\\"\\"\\"\\n    if name not in BOOSTING_CANDIDATES:\\n        raise ValueError(f\\"Unknown boosting recipe: {name}\\")\\n    return Pipeline([\\n        (\\"style\\", StyleFeatures()),\\n        (\\"classifier\\", LGBMClassifier(\\n            **BOOSTING_COMMON, **BOOSTING_CANDIDATES[name],\\n        )),\\n    ])\\n", "ensemble_v2.py": "\\"\\"\\"Fixed-weight combinations of text-only attribution evidence.\\"\\"\\"\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.baseline import LABELS, fit_checked\\nfrom name_that_llm.sparse_v2 import SPARSE_CANDIDATES, build_sparse_pipeline\\nfrom name_that_llm.stylometry import build_style_pipeline\\nfrom name_that_llm.generative import GenerativeConfig, build_generative\\n\\n\\ndef normalize_evidence(scores, *, probabilities=False):\\n    \\"\\"\\"Put different score types on a common per-passage scale, without fitting.\\"\\"\\"\\n    values = np.asarray(scores, dtype=np.float64)\\n    if values.ndim != 2 or values.shape[1] != len(LABELS) or not np.isfinite(values).all():\\n        raise ValueError(\\"Expected finite passage-by-label evidence.\\")\\n    if probabilities:\\n        if np.any(values < 0) or not np.allclose(values.sum(axis=1), 1.0, atol=1e-5):\\n            raise ValueError(\\"Invalid class probabilities.\\")\\n        values = np.log(np.clip(values, 1e-12, 1.0))\\n    centered = values - values.mean(axis=1, keepdims=True)\\n    return centered / np.maximum(centered.std(axis=1, keepdims=True), 1e-8)\\n\\n\\ndef make_component(name):\\n    family, variant = name.split(\\":\\", 1)\\n    if family == \\"sparse\\":\\n        return build_sparse_pipeline(next(c for c in SPARSE_CANDIDATES if c.name == variant))\\n    if family == \\"style\\":\\n        return build_style_pipeline(variant)\\n    if family == \\"generative\\":\\n        return build_generative(GenerativeConfig(variant=variant))\\n    raise ValueError(f\\"Unknown component: {name}\\")\\n\\n\\nclass EvidenceEnsemble(ClassifierMixin, BaseEstimator):\\n    def __init__(self, components, weights):\\n        self.components = components\\n        self.weights = weights\\n\\n    def fit(self, X, y):\\n        weights = np.asarray(self.weights, dtype=float)\\n        if len(weights) != len(self.components) or not len(weights):\\n            raise ValueError(\\"Each component needs a weight.\\")\\n        if not np.isfinite(weights).all() or (weights <= 0).any():\\n            raise ValueError(\\"Weights must be positive and finite.\\")\\n        self.weights_ = weights / weights.sum()\\n        self.models_ = []\\n        self.classes_ = np.asarray(LABELS)\\n        for name in self.components:\\n            model = fit_checked(make_component(name), X, y)\\n            if tuple(model.classes_) != LABELS:\\n                raise ValueError(\\"Component class order differs from competition labels.\\")\\n            self.models_.append(model)\\n        return self\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"weights_\\", \\"classes_\\"])\\n        combined = None\\n        for model, weight in zip(self.models_, self.weights_):\\n            is_probability = hasattr(model, \\"predict_proba\\")\\n            raw = model.predict_proba(X) if is_probability else model.decision_function(X)\\n            evidence = normalize_evidence(raw, probabilities=is_probability)\\n            combined = weight * evidence if combined is None else combined + weight * evidence\\n        return combined\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n", "ensemble_v3.py": "\\"\\"\\"Fixed, passage-local ensembles for the separately recorded v3 experiment.\\"\\"\\"\\n\\nimport numpy as np\\nfrom scipy.special import softmax\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.baseline import LABELS\\nfrom name_that_llm.ensemble_v2 import make_component, normalize_evidence\\n\\n\\ndef make_v3_component(name):\\n    if name.startswith(\\"kernel:\\"):\\n        from name_that_llm.kernel_v3 import build_kernel_pipeline\\n        return build_kernel_pipeline(name.split(\\":\\", 1)[1])\\n    if name.startswith(\\"linear:\\"):\\n        from name_that_llm.linear_v3 import build_linear_pipeline\\n        return build_linear_pipeline(name.split(\\":\\", 1)[1])\\n    if name.startswith(\\"boosting:\\"):\\n        from name_that_llm.boosting_v3 import build_boosting_pipeline\\n        return build_boosting_pipeline(name.split(\\":\\", 1)[1])\\n    if name.startswith(\\"topic:\\"):\\n        from name_that_llm.topic_v3 import build_topic_pipeline\\n        return build_topic_pipeline(name.split(\\":\\", 1)[1])\\n    return make_component(name)\\n\\n\\ndef combine_evidence(evidence, weights, pooling=\\"evidence\\"):\\n    \\"\\"\\"Combine aligned evidence without learning from inference rows.\\"\\"\\"\\n    arrays = [np.asarray(values, dtype=float) for values in evidence]\\n    weights = np.asarray(weights, dtype=float)\\n    if not arrays or len(arrays) != len(weights):\\n        raise ValueError(\\"One weight is required per component.\\")\\n    if not np.isfinite(weights).all() or (weights <= 0).any():\\n        raise ValueError(\\"Weights must be positive and finite.\\")\\n    if pooling not in {\\"evidence\\", \\"probability\\"}:\\n        raise ValueError(\\"Unknown pooling method.\\")\\n    shape = arrays[0].shape\\n    if len(shape) != 2 or shape[1] != len(LABELS):\\n        raise ValueError(\\"Expected passage-by-class scores.\\")\\n    if any(a.shape != shape or not np.isfinite(a).all() for a in arrays):\\n        raise ValueError(\\"Component evidence must be finite and aligned.\\")\\n    if pooling == \\"probability\\":\\n        arrays = [softmax(a, axis=1) for a in arrays]\\n    return sum(a * w for a, w in zip(arrays, weights)) / weights.sum()\\n\\n\\nclass V3Ensemble(ClassifierMixin, BaseEstimator):\\n    def __init__(self, components, weights, pooling=\\"evidence\\"):\\n        self.components = components\\n        self.weights = weights\\n        self.pooling = pooling\\n\\n    def fit(self, X, y, groups=None):\\n        # Check the recipe before starting any expensive fits.\\n        combine_evidence(\\n            [np.zeros((1, len(LABELS))) for _ in self.components],\\n            self.weights, self.pooling,\\n        )\\n        self.classes_ = np.asarray(LABELS)\\n        self.models_ = []\\n        for name in self.components:\\n            model = make_v3_component(name)\\n            if getattr(model, \\"requires_groups\\", False):\\n                if groups is None:\\n                    raise ValueError(\\"This component needs training prompt groups.\\")\\n                model.fit(X, y, groups=groups)\\n            else:\\n                model.fit(X, y)\\n            if tuple(model.classes_) != LABELS:\\n                raise ValueError(\\"Component class order differs from the competition labels.\\")\\n            self.models_.append(model)\\n        return self\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"classes_\\", \\"models_\\"])\\n        evidence = []\\n        for model in self.models_:\\n            probability = hasattr(model, \\"predict_proba\\")\\n            raw = model.predict_proba(X) if probability else model.decision_function(X)\\n            evidence.append(normalize_evidence(raw, probabilities=probability))\\n        return combine_evidence(evidence, self.weights, self.pooling)\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n", "generative.py": "\\"\\"\\"Small, predeclared count-based attribution models for improvement-v2.\\n\\nThe NB-SVM variant implements the log-count-ratio weighting in Wang and\\nManning (2012), with character features and one-versus-rest multiclass fitting.\\nIt does not use their optional coefficient interpolation. This is an adaptation,\\nnot a reproduction of their sentiment experiment.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nimport warnings\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.feature_extraction.text import CountVectorizer\\nfrom sklearn.naive_bayes import ComplementNB, MultinomialNB\\nfrom sklearn.svm import LinearSVC\\nfrom sklearn.utils.validation import check_is_fitted\\n\\n\\nLOGGER = logging.getLogger(\\"name_that_llm.generative\\")\\n\\n\\n@dataclass(frozen=True)\\nclass GenerativeConfig:\\n    \\"\\"\\"Three candidates are fixed before any improvement-v2 validation scores.\\"\\"\\"\\n\\n    variant: str = \\"char_nbsvm\\"\\n    alpha: float = 1.0\\n    c: float = 1.0\\n    min_df: int = 2\\n    max_features: int = 250_000\\n    max_iter: int = 10_000\\n    tolerance: float = 1e-4\\n    seed: int = 42\\n\\n\\nGENERATIVE_CONFIGS = tuple(\\n    GenerativeConfig(variant=variant)\\n    for variant in (\\"char_mnb\\", \\"char_cnb\\", \\"char_nbsvm\\")\\n)\\n\\n\\nclass GenerativeAttribution(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Text-to-score estimator; only ``fit`` constructs a vocabulary.\\n\\n    ``decision_function`` columns follow ``classes_`` (lexical label order).\\n    Scores are uncalibrated. NB scores are per-feature average log evidence;\\n    this preserves each row\'s predicted class while reducing length dependence.\\n    \\"\\"\\"\\n\\n    def __init__(self, config: GenerativeConfig | None = None):\\n        self.config = config\\n\\n    def fit(self, text, labels):\\n        config = self.config or GenerativeConfig()\\n        if config.variant not in {c.variant for c in GENERATIVE_CONFIGS}:\\n            raise ValueError(f\\"Unknown generative variant: {config.variant}\\")\\n        if config.alpha <= 0:\\n            raise ValueError(\\"Smoothing must be strictly positive.\\")\\n        labels = np.asarray(labels)\\n        self.classes_ = np.unique(labels)\\n        if len(self.classes_) < 2:\\n            raise ValueError(\\"At least two labels are required.\\")\\n        self.vectorizer_ = CountVectorizer(\\n            analyzer=\\"char\\",\\n            ngram_range=(3, 5) if config.variant == \\"char_nbsvm\\" else (2, 5),\\n            lowercase=False, strip_accents=None, binary=True,\\n            min_df=config.min_df, max_features=config.max_features,\\n            dtype=np.float64,\\n        )\\n        counts = self.vectorizer_.fit_transform(text)\\n        self.variant_ = config.variant\\n        self.solver_iterations_ = []\\n        if config.variant == \\"char_mnb\\":\\n            self.classifier_ = MultinomialNB(alpha=config.alpha, fit_prior=False)\\n            self.classifier_.fit(counts, labels)\\n        elif config.variant == \\"char_cnb\\":\\n            self.classifier_ = ComplementNB(\\n                alpha=config.alpha, fit_prior=False, norm=True,\\n            )\\n            self.classifier_.fit(counts, labels)\\n        else:\\n            total = np.asarray(counts.sum(axis=0)).ravel()\\n            self.coef_ = np.empty((len(self.classes_), counts.shape[1]))\\n            self.intercept_ = np.empty(len(self.classes_))\\n            for index, label in enumerate(self.classes_):\\n                target = labels == label\\n                positive_count = np.asarray(counts[target].sum(axis=0)).ravel()\\n                positive = positive_count + config.alpha\\n                negative = total - positive_count + config.alpha\\n                ratio = np.log(positive / positive.sum()) - np.log(\\n                    negative / negative.sum()\\n                )\\n                weighted = counts.multiply(ratio).tocsr()\\n                classifier = LinearSVC(\\n                    C=config.c, dual=True, random_state=config.seed,\\n                    tol=config.tolerance, max_iter=config.max_iter,\\n                )\\n                with warnings.catch_warnings(record=True) as caught:\\n                    warnings.simplefilter(\\"always\\")\\n                    classifier.fit(weighted, target)\\n                for warning in caught:\\n                    LOGGER.warning(\\"%s label=%s %s: %s\\", config.variant, label,\\n                                   warning.category.__name__, warning.message)\\n                if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n                    raise RuntimeError(f\\"NB-SVM did not converge for {label}.\\")\\n                # Fold the NB weighting into coefficients for cheap inference.\\n                self.coef_[index] = classifier.coef_[0] * ratio\\n                self.intercept_[index] = classifier.intercept_[0]\\n                self.solver_iterations_.append(int(classifier.n_iter_))\\n        self.n_features_in_ = counts.shape[1]\\n        return self\\n\\n    def decision_function(self, text):\\n        check_is_fitted(self, [\\"classes_\\", \\"vectorizer_\\", \\"variant_\\"])\\n        counts = self.vectorizer_.transform(text)\\n        if self.variant_ == \\"char_nbsvm\\":\\n            scores = np.asarray(counts @ self.coef_.T) + self.intercept_\\n        else:\\n            scores = np.asarray(counts @ self.classifier_.feature_log_prob_.T)\\n            if self.variant_ == \\"char_mnb\\":\\n                scores += self.classifier_.class_log_prior_\\n            denominator = np.maximum(\\n                np.asarray(counts.sum(axis=1)).ravel(), 1.0,\\n            )\\n            scores /= denominator[:, None]\\n        if not np.isfinite(scores).all():\\n            raise ValueError(\\"Model produced non-finite scores.\\")\\n        return scores\\n\\n    def predict(self, text):\\n        return self.classes_[self.decision_function(text).argmax(axis=1)]\\n\\n\\ndef build_generative(config: GenerativeConfig) -> GenerativeAttribution:\\n    return GenerativeAttribution(config)\\n", "kernel_v3.py": "\\"\\"\\"Two fixed nonlinear style-kernel hypotheses for the v3 second stage.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics.pairwise import euclidean_distances\\nfrom sklearn.multiclass import OneVsRestClassifier\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.svm import SVC\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.stylometry import StyleFeatures\\n\\n\\nKERNEL_CANDIDATES = {\\n    \\"style_rbf_c4\\": {\\n        \\"C\\": 4.0, \\"char_weight\\": 0.0, \\"style_weight\\": 1.0,\\n        \\"features\\": \\"465 fixed style features -> training-only StandardScaler\\",\\n        \\"gamma\\": \\"inverse median positive squared training style distance\\",\\n        \\"bandwidth_sample_size\\": 256, \\"selection\\": \\"lowest SHA256(text) values\\",\\n        \\"classifier\\": \\"OneVsRestClassifier(SVC(kernel=precomputed)), n_jobs=1\\",\\n    },\\n    \\"additive_char_style_rbf_c4\\": {\\n        \\"C\\": 4.0, \\"char_weight\\": 0.5, \\"style_weight\\": 0.5,\\n        \\"features\\": \\"char3-5 TFIDF cosine + RBF standardized fixed style features\\",\\n        \\"gamma\\": \\"inverse median positive squared training style distance\\",\\n        \\"bandwidth_sample_size\\": 256, \\"selection\\": \\"lowest SHA256(text) values\\",\\n        \\"classifier\\": \\"OneVsRestClassifier(SVC(kernel=precomputed)), n_jobs=1\\",\\n    },\\n}\\n\\n\\ndef bandwidth_sample_indices(texts, max_samples=256):\\n    \\"\\"\\"Choose training passages reproducibly and independently of row order.\\"\\"\\"\\n    return np.asarray(sorted(\\n        range(len(texts)),\\n        key=lambda index: (hashlib.sha256(texts[index].encode(\\"utf-8\\")).digest(), texts[index]),\\n    )[:max_samples], dtype=int)\\n\\n\\ndef median_style_gamma(styles, indices):\\n    subset = np.asarray(styles, dtype=np.float64)[indices]\\n    squared = euclidean_distances(subset, squared=True)\\n    distances = squared[np.triu_indices(len(subset), k=1)]\\n    positive = distances[distances > 1e-12]\\n    median = float(np.median(positive)) if len(positive) else 1.0\\n    return 1.0 / median\\n\\n\\ndef combine_style_char_kernel(styles_left, styles_right, *, gamma,\\n                              char_left=None, char_right=None, char_weight=0.0):\\n    \\"\\"\\"RBF style similarity plus optional normalized character inner product.\\"\\"\\"\\n    if not np.isfinite(gamma) or gamma <= 0 or not 0 <= char_weight <= 1:\\n        raise ValueError(\\"Invalid kernel bandwidth or mixture weight.\\")\\n    squared = euclidean_distances(styles_left, styles_right, squared=True)\\n    kernel = np.exp(-gamma * squared)\\n    if char_weight:\\n        if char_left is None or char_right is None:\\n            raise ValueError(\\"Character features are required for an additive kernel.\\")\\n        inner = char_left @ char_right.T\\n        if hasattr(inner, \\"toarray\\"):\\n            inner = inner.toarray()\\n        kernel = (1.0 - char_weight) * kernel + char_weight * np.asarray(inner)\\n    if not np.isfinite(kernel).all():\\n        raise ValueError(\\"Non-finite kernel values.\\")\\n    return np.ascontiguousarray(kernel, dtype=np.float64)\\n\\n\\nclass StyleKernelClassifier(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Train-only kernel state; score each text against stored training features.\\"\\"\\"\\n\\n    def __init__(self, C=4.0, char_weight=0.0, bandwidth_sample_size=256, batch_size=256):\\n        self.C = C\\n        self.char_weight = char_weight\\n        self.bandwidth_sample_size = bandwidth_sample_size\\n        self.batch_size = batch_size\\n\\n    def fit(self, X, y):\\n        texts, labels = list(X), np.asarray(y)\\n        if self.C <= 0 or not 0 <= self.char_weight <= 1:\\n            raise ValueError(\\"Invalid classifier configuration.\\")\\n        if self.bandwidth_sample_size < 2 or self.batch_size < 1:\\n            raise ValueError(\\"Invalid bandwidth sample or prediction batch size.\\")\\n        if labels.ndim != 1 or len(texts) != len(labels) or len(texts) < 2:\\n            raise ValueError(\\"Expected aligned texts and labels.\\")\\n        self.extractor_ = StyleFeatures()\\n        self.scaler_ = StandardScaler()\\n        self.training_styles_ = self.scaler_.fit_transform(self.extractor_.transform(texts))\\n        sample = bandwidth_sample_indices(texts, self.bandwidth_sample_size)\\n        self.gamma_ = median_style_gamma(self.training_styles_, sample)\\n        self.n_bandwidth_samples_ = len(sample)\\n        self.training_chars_ = None\\n        if self.char_weight:\\n            self.vectorizer_ = TfidfVectorizer(\\n                analyzer=\\"char\\", ngram_range=(3, 5), min_df=2,\\n                max_features=250_000, lowercase=False, strip_accents=None,\\n                sublinear_tf=True, norm=\\"l2\\", dtype=np.float64,\\n            )\\n            self.training_chars_ = self.vectorizer_.fit_transform(texts)\\n        gram = combine_style_char_kernel(\\n            self.training_styles_, self.training_styles_, gamma=self.gamma_,\\n            char_left=self.training_chars_, char_right=self.training_chars_,\\n            char_weight=self.char_weight,\\n        )\\n        self.classifier_ = OneVsRestClassifier(\\n            SVC(C=self.C, kernel=\\"precomputed\\", tol=1e-3, cache_size=256,\\n                probability=False, max_iter=-1, random_state=42),\\n            n_jobs=1,\\n        )\\n        self.classifier_.fit(gram, labels)\\n        self.classes_ = self.classifier_.classes_\\n        if len(self.classes_) < 2:\\n            raise ValueError(\\"Need at least two classes.\\")\\n        return self\\n\\n    def _kernel(self, texts):\\n        styles = self.scaler_.transform(self.extractor_.transform(texts))\\n        chars = self.vectorizer_.transform(texts) if self.char_weight else None\\n        return combine_style_char_kernel(\\n            styles, self.training_styles_, gamma=self.gamma_,\\n            char_left=chars, char_right=self.training_chars_,\\n            char_weight=self.char_weight,\\n        )\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"classifier_\\", \\"classes_\\", \\"gamma_\\"])\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages.\\")\\n        texts = list(X)\\n        if not texts:\\n            return np.empty((0, len(self.classes_)), dtype=np.float64)\\n        batches = []\\n        for start in range(0, len(texts), self.batch_size):\\n            scores = self.classifier_.decision_function(self._kernel(texts[start:start + self.batch_size]))\\n            if scores.ndim == 1:\\n                scores = np.column_stack([-scores, scores])\\n            batches.append(scores)\\n        return np.concatenate(batches)\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n\\n\\ndef build_kernel_pipeline(name):\\n    if name == \\"style_rbf_c4\\":\\n        return StyleKernelClassifier(C=4.0, char_weight=0.0)\\n    if name == \\"additive_char_style_rbf_c4\\":\\n        return StyleKernelClassifier(C=4.0, char_weight=0.5)\\n    raise ValueError(f\\"Unknown kernel candidate: {name}\\")\\n", "linear_v3.py": "\\"\\"\\"Joint multinomial text/style models for the frozen v3 second stage.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer, StandardScaler\\n\\nfrom name_that_llm.stylometry import StyleFeatures\\n\\n\\nLINEAR_CANDIDATES = {\\n    \\"joint_char_style_c4\\": {\\"C\\": 4.},\\n    \\"joint_char_style_c16\\": {\\"C\\": 16.},\\n}\\n\\nLINEAR_COMMON = {\\n    \\"char_features\\": {\\n        \\"analyzer\\": \\"char\\", \\"ngram_range\\": [1, 5], \\"min_df\\": 2,\\n        \\"max_features\\": 300000, \\"lowercase\\": False, \\"sublinear_tf\\": True,\\n        \\"dtype\\": \\"float64\\",\\n    },\\n    \\"style_features\\": \\"465 StyleFeatures -> StandardScaler -> L2 Normalizer\\",\\n    \\"union_weights\\": {\\"char\\": 1., \\"style\\": .5},\\n    \\"union_postprocess\\": \\"L2 Normalizer\\",\\n    \\"classifier\\": {\\n        \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 1500, \\"tol\\": 1e-4,\\n        \\"loss\\": \\"joint multinomial negative log likelihood\\",\\n        \\"penalty\\": \\"default L2\\", \\"fit_intercept\\": True,\\n    },\\n}\\n\\n\\ndef build_linear_pipeline(name):\\n    \\"\\"\\"Fit character vocabulary, style scaler and joint classifier on fit rows.\\"\\"\\"\\n    if name not in LINEAR_CANDIDATES:\\n        raise ValueError(f\\"Unknown linear recipe: {name}\\")\\n    return Pipeline([\\n        (\\"features\\", FeatureUnion([\\n            (\\"char\\", TfidfVectorizer(\\n                analyzer=\\"char\\", ngram_range=(1, 5), min_df=2,\\n                max_features=300000, lowercase=False, strip_accents=None,\\n                sublinear_tf=True, dtype=np.float64,\\n            )),\\n            (\\"style\\", Pipeline([\\n                (\\"extract\\", StyleFeatures()),\\n                (\\"scale\\", StandardScaler()),\\n                (\\"normalize\\", Normalizer()),\\n            ])),\\n        ], transformer_weights={\\"char\\": 1., \\"style\\": .5})),\\n        (\\"normalize\\", Normalizer()),\\n        (\\"classifier\\", LogisticRegression(\\n            C=LINEAR_CANDIDATES[name][\\"C\\"],\\n            solver=\\"lbfgs\\", max_iter=1500, tol=1e-4,\\n        )),\\n    ])\\n", "sparse_v2.py": "\\"\\"\\"Predeclared sparse-feature variants for the second experiment.\\"\\"\\"\\n\\nfrom dataclasses import asdict, dataclass\\n\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.svm import LinearSVC\\n\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass SparseConfig:\\n    name: str\\n    ngram_min: int\\n    ngram_max: int\\n    c: float\\n    min_df: int = 2\\n    max_features: int = 300_000\\n    seed: int = 42\\n\\n\\nSPARSE_CANDIDATES = (\\n    SparseConfig(\\"char15_c4\\", 1, 5, 4.0),\\n    SparseConfig(\\"char26_c4\\", 2, 6, 4.0),\\n    SparseConfig(\\"char35_c4\\", 3, 5, 4.0),\\n    SparseConfig(\\"char15_c16\\", 1, 5, 16.0),\\n)\\n\\n\\ndef build_sparse_pipeline(config: SparseConfig) -> Pipeline:\\n    return Pipeline([\\n        (\\"features\\", TfidfVectorizer(\\n            analyzer=\\"char\\", ngram_range=(config.ngram_min, config.ngram_max),\\n            lowercase=False, strip_accents=None, min_df=config.min_df,\\n            max_features=config.max_features, sublinear_tf=True, dtype=np.float32,\\n        )),\\n        (\\"classifier\\", LinearSVC(\\n            C=config.c, dual=True, random_state=config.seed,\\n            max_iter=10_000, tol=1e-4,\\n        )),\\n    ])\\n", "stacking_v4.py": "\\"\\"\\"Prompt-group cross-fitted stacking with an explicit audited OOF boundary.\\n\\nThe evaluation driver uses double exclusions for its inner predictions.\\nThis estimator generates ordinary group OOF only inside its own fit dataset.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nimport hashlib\\nimport json\\nimport logging\\nfrom pathlib import Path\\nimport warnings\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, ClassifierMixin\\nfrom sklearn.exceptions import ConvergenceWarning\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.model_selection import GroupKFold\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\nfrom sklearn.utils.validation import check_is_fitted\\nfrom threadpoolctl import threadpool_limits\\n\\nfrom name_that_llm.baseline import LABELS, file_sha256\\nfrom name_that_llm.ensemble_v2 import normalize_evidence\\nfrom name_that_llm.ensemble_v3 import make_v3_component\\n\\nLOGGER = logging.getLogger(\\"nested_v4\\")\\nCOMPONENTS = (\\n    \\"sparse:char15_c4\\",\\n    \\"style:style_hist\\",\\n    \\"style:char_style_svm\\",\\n    \\"linear:joint_char_style_c16\\",\\n)\\nMETA_PARAMETERS = {\\"C\\": .05, \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 2000, \\"tol\\": 1e-4}\\nSOURCE_FILES = (\\n    \\"baseline.py\\", \\"sparse_v2.py\\", \\"stylometry.py\\", \\"linear_v3.py\\",\\n    \\"ensemble_v2.py\\", \\"ensemble_v3.py\\", \\"stacking_v4.py\\",\\n)\\n\\n\\ndef json_hash(value):\\n    return hashlib.sha256(\\n        json.dumps(\\n            value, sort_keys=True, ensure_ascii=False, separators=(\\",\\", \\":\\"),\\n            default=lambda item: item.item() if isinstance(item, np.generic) else item.tolist(),\\n        ).encode()\\n    ).hexdigest()\\n\\n\\ndef array_hash(array):\\n    values = np.ascontiguousarray(array)\\n    return hashlib.sha256(\\n        str(values.dtype).encode() + json.dumps(list(values.shape)).encode() + values.tobytes()\\n    ).hexdigest()\\n\\n\\ndef source_hashes():\\n    root = Path(__file__).resolve().parent\\n    return {name: file_sha256(root / name) for name in SOURCE_FILES}\\n\\n\\ndef recipe_definition():\\n    # Include every effective sklearn parameter, including estimator defaults.\\n    return {\\n        \\"components\\": list(COMPONENTS),\\n        \\"estimator_parameters\\": {\\n            name: {\\n                key: repr(value) for key, value in make_v3_component(name).get_params(deep=True).items()\\n            }\\n            for name in COMPONENTS\\n        },\\n        \\"normalization\\": \\"normalize_evidence per row, log clipped probabilities\\",\\n        \\"meta\\": META_PARAMETERS,\\n        \\"meta_preprocessing\\": \\"StandardScaler fitted on meta training rows only\\",\\n    }\\n\\n\\ndef training_fingerprint(X, y, groups):\\n    return json_hash([list(X), list(y), list(groups)])\\n\\n\\ndef validate_training(X, y, groups):\\n    if isinstance(X, str) or groups is None:\\n        raise ValueError(\\"Pass a sequence of texts and explicit training prompt groups.\\")\\n    texts = np.asarray(X, dtype=object)\\n    targets = np.asarray(y, dtype=object)\\n    prompt_groups = np.asarray(groups, dtype=object)\\n    if any(a.ndim != 1 for a in (texts, targets, prompt_groups)):\\n        raise ValueError(\\"Texts, labels and prompt groups must be one-dimensional.\\")\\n    if not len(texts) or not len(texts) == len(targets) == len(prompt_groups):\\n        raise ValueError(\\"Texts, labels and prompt groups must have equal nonzero length.\\")\\n    if not all(isinstance(t, str) for t in texts):\\n        raise TypeError(\\"All passages must be strings.\\")\\n    if any(g is None or (isinstance(g, float) and np.isnan(g)) for g in prompt_groups):\\n        raise ValueError(\\"Prompt groups cannot be missing.\\")\\n    if set(targets) != set(LABELS):\\n        raise ValueError(\\"All 18 competition classes are required.\\")\\n    return texts, targets, prompt_groups\\n\\n\\ndef validate_fold_assignment(folds, y, groups, *, n_splits=5):\\n    assignment = np.asarray(folds)\\n    targets, prompt_groups = np.asarray(y), np.asarray(groups)\\n    if assignment.ndim != 1 or len(assignment) != len(targets) or len(groups) != len(targets):\\n        raise ValueError(\\"Fold, label and group lengths differ.\\")\\n    if set(assignment) != set(range(n_splits)):\\n        raise ValueError(f\\"Expected folds numbered 0 through {n_splits - 1}.\\")\\n    for group in set(prompt_groups):\\n        if len(set(assignment[prompt_groups == group])) != 1:\\n            raise ValueError(\\"A prompt group crosses folds.\\")\\n    for fold in range(n_splits):\\n        for mask in (assignment == fold, assignment != fold):\\n            if set(targets[mask]) != set(LABELS):\\n                raise ValueError(\\"Every training and held-out fold must contain all classes.\\")\\n    return assignment.astype(np.int64)\\n\\n\\ndef make_group_assignment(X, y, groups, *, n_splits=5, seed=42):\\n    assignment = np.full(len(X), -1, dtype=np.int64)\\n    cv = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)\\n    for fold, (_, valid) in enumerate(cv.split(X, y, groups)):\\n        assignment[valid] = fold\\n    return validate_fold_assignment(assignment, y, groups, n_splits=n_splits)\\n\\n\\ndef fit_component(name, X, y):\\n    LOGGER.info(\\"Fit component=%s rows=%d\\", name, len(X))\\n    model = make_v3_component(name)\\n    with warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(X, y)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(f\\"Component failed to converge: {name}\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Base classifier class order mismatch.\\")\\n    return model\\n\\n\\ndef component_evidence(model, X):\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Base classifier class order mismatch.\\")\\n    probabilities = hasattr(model, \\"predict_proba\\")\\n    values = model.predict_proba(X) if probabilities else model.decision_function(X)\\n    return normalize_evidence(values, probabilities=probabilities)\\n\\n\\ndef evidence_features(evidence):\\n    \\"\\"\\"Turn (rows, four components, eighteen classes) into 72 features.\\"\\"\\"\\n    values = np.asarray(evidence, dtype=np.float64)\\n    if values.ndim != 3 or values.shape[1:] != (len(COMPONENTS), len(LABELS)):\\n        raise ValueError(\\"Expected row-by-component-by-class evidence.\\")\\n    if not np.isfinite(values).all():\\n        raise ValueError(\\"Evidence contains non-finite values.\\")\\n    return values.reshape(len(values), -1)\\n\\n\\ndef fit_meta(evidence, y):\\n    features = evidence_features(evidence)\\n    model = Pipeline([\\n        (\\"scale\\", StandardScaler()),\\n        (\\"classifier\\", LogisticRegression(**META_PARAMETERS)),\\n    ])\\n    with warnings.catch_warnings(record=True) as caught:\\n        warnings.simplefilter(\\"always\\")\\n        model.fit(features, y)\\n    for warning in caught:\\n        LOGGER.warning(\\"%s: %s\\", warning.category.__name__, warning.message)\\n    if any(issubclass(w.category, ConvergenceWarning) for w in caught):\\n        raise RuntimeError(\\"Meta classifier failed to converge.\\")\\n    if tuple(model.classes_) != LABELS:\\n        raise ValueError(\\"Meta classifier class order mismatch.\\")\\n    return model\\n\\n\\n@dataclass(frozen=True)\\nclass AuditedOOF:\\n    \\"\\"\\"An OOF bundle bound to exact training rows, folds, source and recipe.\\n\\n    Constructed by the evaluator after auditing producer provenance, or by\\n    generate_fit_oof. A raw array is deliberately not accepted by fit().\\n    \\"\\"\\"\\n\\n    evidence: np.ndarray\\n    folds: np.ndarray\\n    audit: dict\\n\\n    def validate(self, X, y, groups):\\n        evidence_features(self.evidence)\\n        validate_fold_assignment(self.folds, y, groups)\\n        expected = {\\n            \\"training_fingerprint\\": training_fingerprint(X, y, groups),\\n            \\"evidence_sha256\\": array_hash(self.evidence),\\n            \\"fold_assignment_sha256\\": array_hash(self.folds),\\n            \\"source_hashes\\": source_hashes(),\\n            \\"recipe_sha256\\": json_hash(recipe_definition()),\\n            \\"components\\": list(COMPONENTS),\\n            \\"labels\\": list(LABELS),\\n            \\"group_oof_verified\\": True,\\n        }\\n        if self.evidence.shape[0] != len(X):\\n            raise ValueError(\\"OOF evidence row count differs.\\")\\n        for key, value in expected.items():\\n            if self.audit.get(key) != value:\\n                raise ValueError(f\\"OOF audit mismatch: {key}\\")\\n        if not self.audit.get(\\"provenance\\"):\\n            raise ValueError(\\"OOF provenance is required.\\")\\n        return self\\n\\n\\ndef audited_oof_bundle(evidence, folds, X, y, groups, *, provenance):\\n    \\"\\"\\"Seal an OOF matrix after the caller verifies its fold producer records.\\"\\"\\"\\n    values = np.array(evidence, dtype=np.float64, copy=True)\\n    assignment = validate_fold_assignment(folds, y, groups).copy()\\n    values.flags.writeable = False\\n    assignment.flags.writeable = False\\n    audit = {\\n        \\"training_fingerprint\\": training_fingerprint(X, y, groups),\\n        \\"evidence_sha256\\": array_hash(values),\\n        \\"fold_assignment_sha256\\": array_hash(assignment),\\n        \\"source_hashes\\": source_hashes(),\\n        \\"recipe_sha256\\": json_hash(recipe_definition()),\\n        \\"components\\": list(COMPONENTS),\\n        \\"labels\\": list(LABELS),\\n        \\"group_oof_verified\\": True,\\n        \\"provenance\\": provenance,\\n    }\\n    return AuditedOOF(values, assignment, audit).validate(X, y, groups)\\n\\n\\ndef generate_fit_oof(X, y, groups, *, threads=2):\\n    texts, targets, prompt_groups = validate_training(X, y, groups)\\n    folds = make_group_assignment(texts, targets, prompt_groups)\\n    evidence = np.full((len(texts), len(COMPONENTS), len(LABELS)), np.nan)\\n    provenance = []\\n    with threadpool_limits(limits=threads):\\n        for fold in range(5):\\n            fitting, valid = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)\\n            for component, name in enumerate(COMPONENTS):\\n                model = fit_component(name, texts[fitting], targets[fitting])\\n                evidence[valid, component] = component_evidence(model, texts[valid])\\n            provenance.append({\\n                \\"held_out_fold\\": fold, \\"training_indexes_sha256\\": array_hash(fitting),\\n                \\"held_out_indexes_sha256\\": array_hash(valid),\\n            })\\n    return audited_oof_bundle(\\n        evidence, folds, texts, targets, prompt_groups,\\n        provenance={\\"kind\\": \\"runtime_group_oof\\", \\"seed\\": 42, \\"folds\\": provenance},\\n    )\\n\\n\\nclass NestedStacker(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Four frozen evidence bases and a group cross-fitted logistic meta model.\\"\\"\\"\\n\\n    requires_groups = True\\n\\n    def __init__(self, threads=2):\\n        self.threads = threads\\n\\n    def fit(self, X, y, groups=None, *, fit_oof=None):\\n        texts, targets, prompt_groups = validate_training(X, y, groups)\\n        if not isinstance(self.threads, int) or not 1 <= self.threads <= 2:\\n            raise ValueError(\\"Use one or two training threads.\\")\\n        if fit_oof is None:\\n            fit_oof = generate_fit_oof(texts, targets, prompt_groups, threads=self.threads)\\n        elif not isinstance(fit_oof, AuditedOOF):\\n            raise TypeError(\\"fit_oof must be an AuditedOOF instance, never raw OOF scores.\\")\\n        fit_oof.validate(texts, targets, prompt_groups)\\n        with threadpool_limits(limits=self.threads):\\n            self.meta_ = fit_meta(fit_oof.evidence, targets)\\n            self.models_ = [fit_component(name, texts, targets) for name in COMPONENTS]\\n        self.classes_ = np.asarray(LABELS)\\n        self.fit_oof_audit_ = fit_oof.audit.copy()\\n        self.n_features_in_ = 1\\n        return self\\n\\n    def base_evidence(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages.\\")\\n        texts = list(X)\\n        with threadpool_limits(limits=self.threads):\\n            return np.stack([component_evidence(model, texts) for model in self.models_], axis=1)\\n\\n    def predict_proba(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        return self.meta_.predict_proba(evidence_features(self.base_evidence(X)))\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        return self.meta_.decision_function(evidence_features(self.base_evidence(X)))\\n\\n    def predict(self, X):\\n        check_is_fitted(self, [\\"models_\\", \\"meta_\\", \\"classes_\\"])\\n        return self.classes_[self.predict_proba(X).argmax(axis=1)]\\n", "stylometry.py": "\\"\\"\\"Fixed, text-only style features for the separately recorded v2 experiment.\\n\\nFeature families follow the character/word/function-word measurements in\\nhttps://arxiv.org/html/2104.08530v2, extended with Unicode punctuation and\\nposition-specific discourse features. This is not a reproduction of that paper.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nimport re\\nimport string\\nimport unicodedata\\n\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.ensemble import HistGradientBoostingClassifier\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer, StandardScaler\\nfrom sklearn.svm import LinearSVC\\n\\n\\n# Fixed before fitting; no corpus vocabulary, labels, IDs, or prompt IDs.\\nFUNCTION_WORDS = tuple(dict.fromkeys(\\"\\"\\"\\na an the and or but if because although though while whereas unless until when\\nwhenever where wherever as than that whether since once before after during\\ndespite through throughout within without with into onto upon from of for to\\nat by in on over under between among against about above below beyond across\\naround along toward towards per via not no nor never neither either both each\\nevery any all some many much few little more most less least several enough\\nother another such same own only even also just still already yet again rather\\nquite very too so then thus hence therefore however instead otherwise indeed\\nperhaps maybe almost always often sometimes usually generally typically\\nespecially particularly simply merely really actually essentially ultimately\\nis are was were be been being am do does did done doing have has had having\\ncan could may might must shall should will would need ought\\ni me my mine myself we us our ours ourselves you your yours yourself yourselves\\nhe him his himself she her hers herself it its itself they them their theirs\\nthemselves one ones someone anyone everyone nobody nothing something everything\\nthis these those there here who whom whose which what why how\\n\\"\\"\\".split()))\\n\\nCONNECTIVES = (\\n    \\"for example\\", \\"for instance\\", \\"in other words\\", \\"in practice\\",\\n    \\"in fact\\", \\"as a result\\", \\"on the other hand\\", \\"at the same time\\",\\n    \\"rather than\\", \\"instead of\\", \\"not only\\", \\"but also\\", \\"even if\\",\\n    \\"even though\\", \\"as long as\\", \\"so that\\", \\"such as\\", \\"in contrast\\",\\n    \\"by contrast\\", \\"in short\\", \\"ultimately\\", \\"however\\", \\"therefore\\",\\n    \\"because\\", \\"although\\", \\"while\\", \\"unless\\", \\"whether\\", \\"if\\",\\n)\\nPUNCTUATION = tuple(dict.fromkeys(string.punctuation + \\"\\u2014\\u2013\\u2018\\u2019\\u201c\\u201d\\u2026\\u2212\\u2022\\\\n\\\\t\\"))\\nWORD_RE = re.compile(r\\"[^\\\\W\\\\d_]+(?:[\'\\u2019][^\\\\W\\\\d_]+)*|\\\\d+(?:\\\\.\\\\d+)?\\", re.UNICODE)\\nSENTENCE_RE = re.compile(r\\"(?<=[.!?])(?:[\\\\\\"\'\\u201d\\u2019)]*)\\\\s+\\")\\n\\n\\nclass StyleFeatures(TransformerMixin, BaseEstimator):\\n    \\"\\"\\"Deterministic measurements; fit deliberately learns nothing.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def __sklearn_is_fitted__(self):\\n        return True\\n\\n    def get_feature_names_out(self, input_features=None):\\n        return np.asarray(list(self._measure(\\"\\").keys()), dtype=object)\\n\\n    def transform(self, X):\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages, not one string.\\")\\n        rows = [list(self._measure(text).values()) for text in X]\\n        if not rows:\\n            return np.empty((0, len(self.get_feature_names_out())), dtype=np.float64)\\n        matrix = np.asarray(rows, dtype=np.float64)\\n        if not np.isfinite(matrix).all():\\n            raise ValueError(\\"Non-finite style feature.\\")\\n        return matrix\\n\\n    @staticmethod\\n    def _measure(text):\\n        if not isinstance(text, str):\\n            raise TypeError(\\"StyleFeatures accepts text strings only.\\")\\n        words = WORD_RE.findall(text.lower())\\n        counts = Counter(words)\\n        n = max(1, len(text))\\n        nw = max(1, len(words))\\n        lengths = np.asarray([len(w) for w in words] or [0], dtype=float)\\n        sentences = [s for s in SENTENCE_RE.split(text) if s.strip()]\\n        sentence_lengths = np.asarray(\\n            [len(WORD_RE.findall(s)) for s in sentences] or [0], dtype=float,\\n        )\\n        ns = max(1, len(sentences))\\n        f = {\\n            \\"log_chars\\": np.log1p(len(text)),\\n            \\"log_words\\": np.log1p(len(words)),\\n            \\"log_sentences\\": np.log1p(len(sentences)),\\n            \\"words_per_sentence\\": len(words) / ns,\\n            \\"chars_per_word\\": len(text) / nw,\\n            \\"type_token_ratio\\": len(counts) / nw,\\n            \\"hapax_ratio\\": sum(v == 1 for v in counts.values()) / nw,\\n            \\"repeated_token_ratio\\": sum(v for v in counts.values() if v > 1) / nw,\\n            \\"digits_rate\\": sum(c.isdigit() for c in text) / n,\\n            \\"uppercase_rate\\": sum(c.isupper() for c in text) / n,\\n            \\"letters_rate\\": sum(c.isalpha() for c in text) / n,\\n            \\"whitespace_rate\\": sum(c.isspace() for c in text) / n,\\n            \\"unicode_rate\\": sum(ord(c) > 127 for c in text) / n,\\n            \\"unicode_punctuation_rate\\": sum(\\n                ord(c) > 127 and unicodedata.category(c).startswith(\\"P\\") for c in text\\n            ) / n,\\n            \\"contraction_rate\\": sum(\\"\'\\" in w or \\"\\u2019\\" in w for w in words) / nw,\\n            \\"short_word_rate\\": float(np.mean(lengths <= 3)),\\n            \\"long_word_rate\\": float(np.mean(lengths >= 8)),\\n            \\"question_sentence_rate\\": sum(\\"?\\" in s for s in sentences) / ns,\\n            \\"exclamation_sentence_rate\\": sum(\\"!\\" in s for s in sentences) / ns,\\n            \\"sentence_length_alternation\\": float(np.abs(np.diff(sentence_lengths)).mean())\\n            if len(sentence_lengths) > 1 else 0.,\\n        }\\n        for prefix, values in ((\\"word_length\\", lengths), (\\"sentence_words\\", sentence_lengths)):\\n            for name, value in (\\n                (\\"mean\\", np.mean(values)), (\\"std\\", np.std(values)),\\n                (\\"min\\", np.min(values)), (\\"max\\", np.max(values)),\\n                (\\"q25\\", np.quantile(values, .25)), (\\"q50\\", np.quantile(values, .5)),\\n                (\\"q75\\", np.quantile(values, .75)),\\n            ):\\n                f[f\\"{prefix}_{name}\\"] = float(value)\\n        for length in range(1, 16):\\n            f[f\\"word_length_bin_{length}\\"] = float(np.mean(lengths == length))\\n        for char in PUNCTUATION:\\n            key = f\\"punct_u{ord(char):04x}\\"\\n            f[f\\"{key}_rate\\"] = text.count(char) / nw\\n            f[f\\"{key}_present\\"] = float(char in text)\\n        for char in string.ascii_lowercase:\\n            f[f\\"letter_{char}_rate\\"] = text.lower().count(char) / n\\n        for word in FUNCTION_WORDS:\\n            f[f\\"fw_{word}\\"] = counts[word] / nw\\n        lower = \\" \\" + \\" \\".join(words) + \\" \\"\\n        first = \\" \\" + \\" \\".join(words[:15]) + \\" \\"\\n        last = \\" \\" + \\" \\".join(words[-15:]) + \\" \\"\\n        for connective in CONNECTIVES:\\n            needle = \\" \\" + connective + \\" \\"\\n            f[f\\"connective_{connective}\\"] = lower.count(needle) / nw\\n            f[f\\"opening_{connective}\\"] = float(needle in first)\\n            f[f\\"ending_{connective}\\"] = float(needle in last)\\n        for position, sentence in (\\n            (\\"opening\\", sentences[0] if sentences else \\"\\"),\\n            (\\"ending\\", sentences[-1] if sentences else \\"\\"),\\n        ):\\n            sw = WORD_RE.findall(sentence.lower())\\n            f[f\\"{position}_length_share\\"] = len(sw) / nw\\n            f[f\\"{position}_question\\"] = float(\\"?\\" in sentence)\\n            f[f\\"{position}_colon\\"] = float(\\":\\" in sentence)\\n            f[f\\"{position}_dash\\"] = float(\\"\\u2014\\" in sentence)\\n            f[f\\"{position}_first_person\\"] = sum(w in {\\"i\\", \\"we\\", \\"our\\", \\"my\\"} for w in sw) / max(1, len(sw))\\n            f[f\\"{position}_second_person\\"] = sum(w in {\\"you\\", \\"your\\"} for w in sw) / max(1, len(sw))\\n        return f\\n\\n\\nCANDIDATES = {\\n    \\"style_logistic\\": {\\n        \\"features\\": \\"fixed StyleFeatures -> StandardScaler\\",\\n        \\"classifier\\": \\"LogisticRegression\\", \\"C\\": .05,\\n        \\"solver\\": \\"lbfgs\\", \\"max_iter\\": 1500,\\n    },\\n    \\"style_hist\\": {\\n        \\"features\\": \\"fixed StyleFeatures\\", \\"classifier\\": \\"HistGradientBoostingClassifier\\",\\n        \\"max_iter\\": 160, \\"max_leaf_nodes\\": 7, \\"learning_rate\\": .07,\\n        \\"l2_regularization\\": 3., \\"min_samples_leaf\\": 25,\\n        \\"early_stopping\\": False, \\"random_state\\": 42,\\n    },\\n    \\"char_style_svm\\": {\\n        \\"features\\": \\"char TFIDF (3,5), min_df2, max_features250000, case-sensitive\\"\\n        \\" + .5 * L2-normalized standardized style; concatenate and L2-normalize\\",\\n        \\"classifier\\": \\"LinearSVC\\", \\"C\\": 1., \\"max_iter\\": 10000,\\n        \\"dual\\": True, \\"random_state\\": 42,\\n    },\\n}\\n\\n\\ndef build_style_pipeline(name):\\n    style = StyleFeatures()\\n    if name == \\"style_logistic\\":\\n        return Pipeline([\\n            (\\"style\\", style), (\\"scale\\", StandardScaler()),\\n            (\\"classifier\\", LogisticRegression(C=.05, solver=\\"lbfgs\\", max_iter=1500)),\\n        ])\\n    if name == \\"style_hist\\":\\n        return Pipeline([\\n            (\\"style\\", style),\\n            (\\"classifier\\", HistGradientBoostingClassifier(\\n                max_iter=160, max_leaf_nodes=7, learning_rate=.07,\\n                l2_regularization=3., min_samples_leaf=25,\\n                early_stopping=False, random_state=42,\\n            )),\\n        ])\\n    if name == \\"char_style_svm\\":\\n        return Pipeline([\\n            (\\"features\\", FeatureUnion([\\n                (\\"char\\", TfidfVectorizer(\\n                    analyzer=\\"char\\", ngram_range=(3, 5), min_df=2,\\n                    max_features=250_000, lowercase=False, strip_accents=None,\\n                    sublinear_tf=True, dtype=np.float32,\\n                )),\\n                (\\"style\\", Pipeline([\\n                    (\\"extract\\", style), (\\"scale\\", StandardScaler()),\\n                    (\\"normalize\\", Normalizer()),\\n                ])),\\n            ], transformer_weights={\\"char\\": 1., \\"style\\": .5})),\\n            (\\"normalize\\", Normalizer()),\\n            (\\"classifier\\", LinearSVC(\\n                C=1., max_iter=10000, dual=True, random_state=42, tol=1e-4,\\n            )),\\n        ])\\n    raise ValueError(name)\\n\\n", "topic_v3.py": "\\"\\"\\"Training-only prompt nuisance removal and fixed content masking for v3.\\n\\nThe masking experiment is motivated by arXiv:2104.08530, not a reproduction.\\nPrompt-centered ridge is a separately derived hypothesis: train within prompts,\\nthen predict individual passages relative to the training global feature mean.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport re\\n\\nimport numpy as np\\nfrom scipy.linalg import solve\\nfrom sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import FeatureUnion, Pipeline\\nfrom sklearn.preprocessing import Normalizer, StandardScaler\\nfrom sklearn.svm import LinearSVC\\nfrom sklearn.utils.validation import check_is_fitted\\n\\nfrom name_that_llm.stylometry import FUNCTION_WORDS, StyleFeatures\\n\\n\\nTOPIC_CANDIDATES = {\\n    \\"centered_ridge_half\\": {\\n        \\"centering\\": 0.5, \\"ridge_alpha\\": 1.0,\\n        \\"features\\": \\"raw char3-5 TFIDF + 0.5 normalized standardized style\\",\\n    },\\n    \\"centered_ridge_full\\": {\\n        \\"centering\\": 1.0, \\"ridge_alpha\\": 1.0,\\n        \\"features\\": \\"raw char3-5 TFIDF + 0.5 normalized standardized style\\",\\n    },\\n    \\"masked_char_style_svm\\": {\\n        \\"mask\\": \\"retain fixed function words; other letters \'*\'; digits \'#\'\\",\\n        \\"C\\": 1.0,\\n        \\"features\\": \\"masked char3-5 TFIDF + 0.5 normalized standardized raw style\\",\\n    },\\n}\\n\\nMASK_TOKEN = re.compile(r\\"[^\\\\W\\\\d_]+(?:[\'\\u2019][^\\\\W\\\\d_]+)*|\\\\d+\\", re.UNICODE)\\nKEEP_WORDS = frozenset(FUNCTION_WORDS)\\n\\n\\nclass MaskContent(TransformerMixin, BaseEstimator):\\n    \\"\\"\\"Preserve punctuation, function words and word lengths, deterministically.\\"\\"\\"\\n\\n    def fit(self, X, y=None):\\n        return self\\n\\n    def __sklearn_is_fitted__(self):\\n        return True\\n\\n    @staticmethod\\n    def mask(text):\\n        if not isinstance(text, str):\\n            raise TypeError(\\"Expected a passage string.\\")\\n\\n        def replacement(match):\\n            word = match.group()\\n            if word.lower() in KEEP_WORDS:\\n                return word\\n            return \\"\\".join(\\"#\\" if c.isdigit() else \\"*\\" if c.isalpha() else c for c in word)\\n\\n        return MASK_TOKEN.sub(replacement, text)\\n\\n    def transform(self, X):\\n        if isinstance(X, str):\\n            raise ValueError(\\"Expected a sequence of passages.\\")\\n        return [self.mask(text) for text in X]\\n\\n\\ndef build_topic_features(masked=False):\\n    char = TfidfVectorizer(\\n        analyzer=\\"char\\", ngram_range=(3, 5), min_df=2, max_features=250_000,\\n        lowercase=False, strip_accents=None, sublinear_tf=True, dtype=np.float64,\\n    )\\n    if masked:\\n        char = Pipeline([(\\"mask\\", MaskContent()), (\\"tfidf\\", char)])\\n    return Pipeline([\\n        (\\"features\\", FeatureUnion([\\n            (\\"char\\", char),\\n            (\\"style\\", Pipeline([\\n                (\\"extract\\", StyleFeatures()), (\\"scale\\", StandardScaler()),\\n                (\\"normalize\\", Normalizer()),\\n            ])),\\n        ], transformer_weights={\\"char\\": 1.0, \\"style\\": 0.5})),\\n        (\\"normalize\\", Normalizer()),\\n    ])\\n\\n\\ndef center_by_training_groups(values, group_indices, strength):\\n    \\"\\"\\"Apply A = I - strength*G - (1-strength)*J to rows.\\n\\n    G averages rows inside each training prompt; J is the training global mean.\\n    These are symmetric projectors and A never uses labels or held-out rows.\\n    \\"\\"\\"\\n    matrix = np.asarray(values, dtype=np.float64)\\n    result = matrix - (1.0 - strength) * matrix.mean(axis=0, keepdims=True)\\n    for indices in group_indices:\\n        result[indices] -= strength * matrix[indices].mean(axis=0, keepdims=True)\\n    return result\\n\\n\\nclass PromptCenteredRidge(ClassifierMixin, BaseEstimator):\\n    \\"\\"\\"Fit ridge on prompt-centered training features; infer with a global mean.\\n\\n    Fit requires training prompt groups. Prediction deliberately cannot accept\\n    groups: a passage is scored without reference to any other test passage.\\n    The Gram implementation avoids densifying the sparse feature matrix.\\n    \\"\\"\\"\\n\\n    requires_groups = True\\n\\n    def __init__(self, strength=1.0, alpha=1.0):\\n        self.strength = strength\\n        self.alpha = alpha\\n\\n    def fit(self, X, y, groups=None):\\n        if not 0.0 <= self.strength <= 1.0 or self.alpha <= 0:\\n            raise ValueError(\\"Invalid centering strength or ridge regularization.\\")\\n        texts, labels = list(X), np.asarray(y)\\n        if groups is None:\\n            raise ValueError(\\"Training prompt groups are required.\\")\\n        groups = np.asarray(groups)\\n        if labels.ndim != 1 or groups.ndim != 1 or not len(texts) == len(labels) == len(groups):\\n            raise ValueError(\\"Texts, labels and groups must be aligned one-dimensional arrays.\\")\\n        self.classes_, label_indices = np.unique(labels, return_inverse=True)\\n        if len(self.classes_) < 2:\\n            raise ValueError(\\"Need at least two classes.\\")\\n        _, group_codes = np.unique(groups, return_inverse=True)\\n        group_indices = [np.flatnonzero(group_codes == code) for code in np.unique(group_codes)]\\n        if len(group_indices) < 2:\\n            raise ValueError(\\"Need at least two training prompt groups.\\")\\n        self.features_ = build_topic_features()\\n        features = self.features_.fit_transform(texts, labels)\\n        targets = np.eye(len(self.classes_))[label_indices]\\n        prior = targets.mean(axis=0)\\n        targets -= prior\\n        gram = (features @ features.T).toarray()\\n        centered = center_by_training_groups(gram, group_indices, self.strength)\\n        centered = center_by_training_groups(centered.T, group_indices, self.strength).T\\n        # Roundoff can leave tiny asymmetry; restoring symmetry precedes the SPD solve.\\n        centered = (centered + centered.T) * 0.5\\n        centered.flat[::len(texts) + 1] += self.alpha\\n        dual = solve(centered, targets, assume_a=\\"pos\\", check_finite=True)\\n        adjusted_dual = center_by_training_groups(dual, group_indices, self.strength)\\n        self.coef_ = np.asarray(features.T @ adjusted_dual).T\\n        feature_mean = np.asarray(features.mean(axis=0)).ravel()\\n        self.intercept_ = prior - self.coef_ @ feature_mean\\n        self.n_training_groups_ = len(group_indices)\\n        if not np.isfinite(self.coef_).all() or not np.isfinite(self.intercept_).all():\\n            raise ValueError(\\"Non-finite fitted parameters.\\")\\n        return self\\n\\n    def decision_function(self, X):\\n        check_is_fitted(self, [\\"features_\\", \\"classes_\\", \\"coef_\\", \\"intercept_\\"])\\n        return np.asarray(self.features_.transform(X) @ self.coef_.T) + self.intercept_\\n\\n    def predict(self, X):\\n        return self.classes_[self.decision_function(X).argmax(axis=1)]\\n\\n\\ndef build_topic_pipeline(name):\\n    if name == \\"centered_ridge_half\\":\\n        return PromptCenteredRidge(strength=0.5, alpha=1.0)\\n    if name == \\"centered_ridge_full\\":\\n        return PromptCenteredRidge(strength=1.0, alpha=1.0)\\n    if name == \\"masked_char_style_svm\\":\\n        return Pipeline([\\n            (\\"features\\", build_topic_features(masked=True)),\\n            (\\"classifier\\", LinearSVC(C=1.0, max_iter=10000, dual=True, random_state=42, tol=1e-4)),\\n        ])\\n    raise ValueError(f\\"Unknown topic candidate: {name}\\")\\n"}')
package = Path.cwd() / 'name_that_llm'
package.mkdir(exist_ok=True)
for name, source in sources.items():
    (package / name).write_text(source)
sys.path.insert(0, str(package.parent))


In [ ]:
import logging
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from name_that_llm.baseline import LABELS, load_data, validate_submission, runtime_versions
from name_that_llm.ensemble_v2 import normalize_evidence
from name_that_llm.stacking_v4 import NestedStacker, evidence_features

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
recipe = json.loads('{"stack_weight": 0.5, "v3_weight": 0.5}')
candidates = [Path("/kaggle/input/competitions/name-that-llm"), Path("/kaggle/input/name-that-llm")]
if os.environ.get("NAME_THAT_LLM_DATA_DIR"):
    candidates = [Path(os.environ["NAME_THAT_LLM_DATA_DIR"])]
data_dir = next((p for p in candidates if (p / "train.csv").is_file() and (p / "test.csv").is_file()), None)
if data_dir is None:
    raise FileNotFoundError("Attach the name-that-llm competition data.")
train, test = load_data(data_dir)
print(json.dumps({"versions": runtime_versions(), "recipe": recipe,
                  "train_rows": len(train), "test_rows": len(test)}, indent=2))
with threadpool_limits(limits=2):
    model = NestedStacker(threads=2).fit(train["text"], train["model"], groups=train["prompt_id"])
    base = model.base_evidence(test["text"])
    stack = normalize_evidence(model.meta_.predict_proba(evidence_features(base)), probabilities=True)
    scores = recipe["stack_weight"] * stack + recipe["v3_weight"] * base[:, [0, 1, 3], :].mean(axis=1)
    submission = pd.DataFrame({"id": test["id"].to_numpy(), "model": np.asarray(LABELS)[scores.argmax(axis=1)]})
validate_submission(test, submission)
submission.to_parquet("submission.parquet", index=False)
validate_submission(test, pd.read_parquet("submission.parquet"))
Path("training_audit.json").write_text(json.dumps(model.fit_oof_audit_, indent=2) + "\n")
print(f"Verified submission.parquet: {len(submission)} rows")